In [17]:
# Scientific computing.
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import IPython
import random

# Resolve Utilities path from the actual notebook location.
# See NaiveBayes_TD_IDF for mroe information.
try:
    notebook_path = Path(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"])
    utilities_path = notebook_path.parent.parent / "Utilities"
except KeyError:
    utilities_path = Path().resolve() / "Notebooks" / "Utilities"

sys.path.append(str(utilities_path))

from SI_Utilities import (
    load_tfidf_vectorizer,
    get_agreement_index,
    prepare_model_data_tfidf,
    Leader_R_Cols,
    Primary_Pairs,
)
from Model_Utilities import (
    run_tfidf_lr,
    get_top_features_lr,
    build_summary_tfidf_df,
)

# Machine learning.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [18]:
# A copy-paste from NaiveBayes_TF_IDF notebook.
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

# Compute averaged leader score.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [21]:
# Define the temporal holdout before touching any modeling.
# This dataset is temporal in nature, and should be treated with temporality in mind.

# Fall 2024 and Spring 2024 are the holdout years.
holdout_mask = (
    ((Survey_df["Year"] == 2024) & (Survey_df["Semester"] == "Fall")) |
    (Survey_df["Year"] == 2025)
)
train_mask = ~holdout_mask

# About a 22.9% holdout.
Survey_Train_df = Survey_df[train_mask].copy()
Survey_Holdout_df = Survey_df[holdout_mask].copy()

print(f"Training rows: {len(Survey_Train_df)}")
print(f"Holdout rows: {len(Survey_Holdout_df)}")
print(f"Holdout %: {round(len(Survey_Holdout_df) / len(Survey_df) * 100, 1)}%")
print()

Training rows: 26619
Holdout rows: 7893
Holdout %: 22.9%



In [23]:
# Build lemma strings on the full dataset first, then split.
# The vectorizer was fitted on the full dataset in Text_Parsing.
# No leakage assured: The vectorizer was fitted in Text_Parsing.ipynb on the full dataset.
# Only learns vocabulary and IDF weights, and not target values (positive and negative).
Text_Cols = list(set(t_col for t_col, _ in Primary_Pairs))
for col in Text_Cols:
    Survey_df[col + "_lemma_str"] = Survey_df[col + "_lemmas"].apply(
        lambda x: " ".join(x) if len(x) > 0 else ""
    )
    # Propagate to train and holdout splits.
    Survey_Train_df[col + "_lemma_str"] = Survey_df.loc[train_mask, col + "_lemma_str"]
    Survey_Holdout_df[col + "_lemma_str"] = Survey_df.loc[holdout_mask, col + "_lemma_str"]

# Load vectorizers.
Vectorizers = {col: load_tfidf_vectorizer(col, Project_Root) for col in Text_Cols}
print(f"Loaded {len(Vectorizers)} vectorizers.")

Loaded 4 vectorizers.
